# SimJEB Constraint Visualization

This notebook visualizes the constraint points for the SimJEB (Jet Engine Bracket) problem.

**Constraints:**
- **Outside envelope (red)**: Points where SDF should be positive (no material)
- **Inside envelope (blue)**: Points where material CAN exist
- **Interface (green)**: Boundary points where SDF = 0 with prescribed normals
- **Far outside (orange)**: Points far from envelope to strongly enforce SDF > 0

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

torch.set_default_device('cpu')
print("Imports complete")

In [ ]:
from GINN.problems.problem_simjeb import ProblemSimjeb

# Create the SimJEB problem
problem = ProblemSimjeb(
    nx=3,
    simjeb_root_dir='../GINN/simJEB/data',
    n_points_envelope=3000,
    n_points_interfaces=1000,
    n_points_domain=5000,
    n_points_normals=1000,
    nf_is_density=False  # SDF mode
)
print("SimJEB problem loaded!")

In [ ]:
# Get all constraint points
print("Available constraint point sets:")
for key, pts in problem.constr_pts_dict.items():
    print(f"  {key}: {pts.shape}")

print(f"\nBounds: {problem.bounds}")

In [ ]:
# Extract constraint points
pts_far_outside = problem.constr_pts_dict['far_outside_envelope']
pts_outside = problem.constr_pts_dict['outside_envelope']
pts_around_if = problem.constr_pts_dict['envelope_around_interface']
pts_inside = problem.constr_pts_dict['inside_envelope']
pts_interface = problem.constr_pts_dict['interface']
pts_domain = problem.constr_pts_dict['domain']

print(f"Far outside envelope: {pts_far_outside.shape}")
print(f"Outside envelope: {pts_outside.shape}")
print(f"Around interface: {pts_around_if.shape}")
print(f"Inside envelope: {pts_inside.shape}")
print(f"Interface: {pts_interface.shape}")
print(f"Domain: {pts_domain.shape}")

## Matplotlib 3D Visualization

In [ ]:
# Create 3D visualization with matplotlib
fig = plt.figure(figsize=(18, 5))

# Plot 1: All envelope constraints
ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(pts_far_outside[::5, 0], pts_far_outside[::5, 1], pts_far_outside[::5, 2], 
            c='orange', s=1, alpha=0.3, label='Far outside')
ax1.scatter(pts_outside[::5, 0], pts_outside[::5, 1], pts_outside[::5, 2], 
            c='red', s=1, alpha=0.3, label='Outside envelope')
ax1.scatter(pts_inside[::10, 0], pts_inside[::10, 1], pts_inside[::10, 2], 
            c='blue', s=1, alpha=0.3, label='Inside envelope')
ax1.set_title('Envelope Constraints')
ax1.legend(loc='upper left', fontsize=7)
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

# Plot 2: Inside envelope (shows where shape CAN exist)
ax2 = fig.add_subplot(132, projection='3d')
ax2.scatter(pts_inside[::5, 0], pts_inside[::5, 1], pts_inside[::5, 2], 
            c='blue', s=2, alpha=0.5)
ax2.set_title('Inside Envelope\n(Design Region)')
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')

# Plot 3: Interface points (attachment boundaries)
ax3 = fig.add_subplot(133, projection='3d')
ax3.scatter(pts_interface[:, 0], pts_interface[:, 1], pts_interface[:, 2], 
            c='green', s=5, alpha=0.8)
ax3.set_title('Interface Points\n(Attachment Boundaries)')
ax3.set_xlabel('X'); ax3.set_ylabel('Y'); ax3.set_zlabel('Z')

plt.suptitle('SimJEB Jet Engine Bracket Constraints', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Interactive 3D with k3d

In [ ]:
try:
    import k3d
    
    fig = k3d.plot(height=700)
    
    # Far outside envelope (orange)
    fig += k3d.points(pts_far_outside[::3].astype(np.float32), 
                      color=0xf39c12, point_size=0.01, name='Far outside')
    
    # Outside envelope (red)
    fig += k3d.points(pts_outside[::3].astype(np.float32), 
                      color=0xe74c3c, point_size=0.01, name='Outside envelope')
    
    # Around interface (yellow)
    fig += k3d.points(pts_around_if[::3].astype(np.float32), 
                      color=0xf1c40f, point_size=0.01, name='Around interface')
    
    # Inside envelope (blue)
    fig += k3d.points(pts_inside[::5].astype(np.float32), 
                      color=0x3498db, point_size=0.01, name='Inside envelope')
    
    # Interface points (green)
    fig += k3d.points(pts_interface.astype(np.float32), 
                      color=0x2ecc71, point_size=0.02, name='Interface')
    
    fig.display()
    print("Interactive 3D visualization loaded!")
    print("Toggle layers in the menu on the right.")
    
except ImportError:
    print("k3d not installed. Using matplotlib only.")
    print("Install with: pip install k3d")

## View the Envelope Mesh

In [ ]:
# Visualize the envelope mesh
try:
    import k3d
    
    fig = k3d.plot(height=600)
    
    # Envelope mesh (the outer boundary)
    verts = problem.mesh_env.vertices.astype(np.float32)
    faces = problem.mesh_env.faces.astype(np.uint32)
    fig += k3d.mesh(verts, faces, color=0xcccccc, side='double', 
                    opacity=0.3, name='Envelope mesh')
    
    # Interface mesh (attachment points)
    verts_if = problem.mesh_if.vertices.astype(np.float32)
    faces_if = problem.mesh_if.faces.astype(np.uint32)
    fig += k3d.mesh(verts_if, faces_if, color=0x2ecc71, side='double',
                    name='Interface mesh')
    
    fig.display()
    print("Envelope mesh (gray) and Interface mesh (green)")
    
except ImportError:
    print("k3d not available for mesh visualization")
except Exception as e:
    print(f"Could not load meshes: {e}")

## Constraint Breakdown

The SimJEB problem uses these constraint types:

| Constraint | Points | Loss Applied | Purpose |
|------------|--------|--------------|----------|
| `far_outside_envelope` | Uniformly in outer bounding box | `envelope_loss` (SDF > 0) | Force SDF positive far from design |
| `outside_envelope` | Near envelope surface | `envelope_loss` (SDF > 0) | No material outside envelope |
| `envelope_around_interface` | 10mm buffer around interfaces | `envelope_loss` | Extra enforcement near boundaries |
| `inside_envelope` | Inside the envelope mesh | Training domain | Where shape CAN exist |
| `interface` | On attachment surfaces | `interface_loss` (SDF = 0) + `normal_loss` | Exact boundary with prescribed normals |

In [ ]:
# Show interface normals
interface_pts_sample, interface_normals = problem._interface_constraints[0].get_sampled_points(200)

print(f"Interface points sample: {interface_pts_sample.shape}")
print(f"Interface normals: {interface_normals.shape}")
print(f"\nNormal vectors (first 5):")
for i in range(5):
    pt = interface_pts_sample[i].cpu().numpy()
    n = interface_normals[i].cpu().numpy()
    print(f"  Point {pt} -> Normal {n}")

In [ ]:
# Visualize interface points with normal vectors
try:
    import k3d
    
    fig = k3d.plot(height=600)
    
    # Interface points
    pts = interface_pts_sample.cpu().numpy().astype(np.float32)
    normals = interface_normals.cpu().numpy().astype(np.float32)
    
    fig += k3d.points(pts, color=0x2ecc71, point_size=0.02, name='Interface points')
    
    # Normal vectors as lines
    scale = 0.05  # Length of normal arrows
    for i in range(0, len(pts), 5):  # Every 5th point
        origin = pts[i]
        end = pts[i] + normals[i] * scale
        fig += k3d.line([origin, end], color=0xe74c3c, width=0.003)
    
    fig.display()
    print("Green = interface points, Red lines = normal vectors")
    
except ImportError:
    # Matplotlib fallback
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    pts = interface_pts_sample.cpu().numpy()
    normals = interface_normals.cpu().numpy()
    
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c='green', s=10)
    ax.quiver(pts[::5, 0], pts[::5, 1], pts[::5, 2],
              normals[::5, 0], normals[::5, 1], normals[::5, 2],
              length=0.05, color='red', alpha=0.7)
    ax.set_title('Interface Points with Normal Vectors')
    plt.show()

## Compare with LEGO Problem Structure

| Aspect | SimJEB | LEGO 1xN |
|--------|--------|----------|
| Envelope source | Pre-computed mesh (`411_for_envelope.obj`) | Procedurally generated from dimensions |
| Interface source | Pre-computed mesh (`interfaces.stl`) | Procedurally generated cylinders |
| Point sampling | Loaded from `.npy` files | Computed on-the-fly |
| Complexity | Complex 3D organic shape | Simple geometric primitives |
| Normalization | Centered and scaled | Fixed stud-based scale |